In [13]:
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型
在LangChain中，发送给LLM的消息，LLM返回的消息都被统一封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色创建了对应的子类：
- SystemMessage: role是system，代表系统消息，用于设定模型和交互背景
- HumanMessage: role是user，代表用户输入的消息
- AIMessage: role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage: role是tool，代表工具调用时产生的结果

我们可以直接使用这些Message类型来发送消息。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage


# 定义工具
@tool(description='查询天气工具')
def get_weather(location: str) -> str:
  return f'当前{location}的天气是Sunny'

# 创建智能体
agent = create_agent(model='deepseek-chat', tools=[get_weather])

# 调用智能体
response = agent.invoke({
  'messages': [
    # {'role':'system','content':'你是个东北话嘎嘎溜的AI智能助手'},
    # {'role':'user','content':'你好，我的小猫11岁了'},
    # {'role':'assistant','content':'已经可以说是老大哥了啊'},
    # {'role':'user','content':'我想带他出去散步，广州天气怎么样现在'}
    # 使用子类创建消息类型
    SystemMessage('你是个大同话特别溜的AI智能助手'),
    HumanMessage('你好，我的小猫11岁了'),
    AIMessage('已经可以说是老大哥了啊'),
    HumanMessage('我想带他出去散步，广州天气怎么样现在')
  ]
})

for msg in response['messages']:
  msg.pretty_print()

# 多模态消息
LangChain也支持向模型发送多模态消息，比如图片、视频、音频等，但前提必须是支持多模态的大模型。
现在以qwen3.5-plus为例
## 在线图片
将在线图片发送给模型

In [15]:
import os
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

model = init_chat_model(
  model='qwen3.5-omni-plus',
  model_provider='openai',
  base_url=os.getenv('DASHSCOPE_BASE_URL'),
  api_key=os.getenv('DASHSCOPE_API_KEY')
)

# 创建Agent
agent = create_agent(model=model)

In [ ]:
# 准备多模态消息
message = HumanMessage([
    {'type': 'text', 'text': '描述一下这张图片的内容。'},
    {'type': 'image', 'url': 'https://pic.rmb.bdstatic.com/bjh/events/5065d3954834a6bc88c532fde38c9a6a3707.png@h_1280'}
  ])

In [ ]:
stream = agent.stream(
  {'messages': [message]},
  stream_mode='messages'
)

for chunk, metadata in stream:
  if chunk.content:
    print(chunk.content, end='', flush=True)

## 本地图片数据
有时候用户想上传本地图片数据，而不是url地址。我们需要将图片数据转换为base64字符串再发送给模型。
接下来会模拟图片上传、转换的过程。
首先，安装一个上传组件，用于模拟图片上传。
`uv add ipywidgets`

In [ ]:
uv add ipywidgets

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = wd.FileUpload(accept='*', multiple=False, button_style='primary')
display(uploader)

# print(uploader.value)

In [ ]:
# 将图片转为base64字符串
import base64
# 获取第一个上传文件(唯一的一个)
upload_file = uoloader.value[0]
# 转为图片内存
content_memory = upload_file['content']
# 转为字节
img_bytes = bytes(content_memory)
# 转base64
img = base64.b64encode(img_bytes).decode('utf-8')

In [ ]:
# 组织多模态消息
multimodal = HumanMessage(content=[
  {
    'type': 'image',
    'base64': img,
    'mime_type': 'image/jpeg'
  },
  {'type':'text','text':'给我讲讲图片中的城市'}
])

response = agent.stream(
  {'messages': [multimodal]},
  stream_mode='messages'
)

for chunk,metadata in response:
  print(chunk.content, end='', flush=True)